In [ ]:
import os, random, json, time, shutil, glob
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device, "| PyTorch :", torch.__version__)
if device.type == "cuda": print(torch.cuda.get_device_name(0))

In [ ]:
hits = glob.glob("/kaggle/input/**/train.csv", recursive=True)
print("train.csv trouvé :", hits)
DATA_ROOT = os.path.dirname(hits[0])
print("DATA_ROOT :", DATA_ROOT, os.listdir(DATA_ROOT))

CKPT_DIR = "/kaggle/working/checkpoints"
OUT_DIR  = "/kaggle/working/outputs"
for d in (CKPT_DIR, OUT_DIR): os.makedirs(d, exist_ok=True)

CLASSES = ["NORMAL", "PNEUMONIA"]            # NORMAL=0, PNEUMONIA=1
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
PATH_COL, LABEL_COL, PID_COL = "image_path", "label", "patient_id"   # à adapter après la cellule 3

In [ ]:
train_df = pd.read_csv(f"{DATA_ROOT}/train.csv")
val_df   = pd.read_csv(f"{DATA_ROOT}/val.csv")
print(train_df.columns.tolist())
display(train_df.head()); display(val_df.head())
print(train_df[LABEL_COL].unique())

In [ ]:
def to_idx(x):
    if isinstance(x, str):
        s = x.strip().upper()
        return CLASS_TO_IDX[s] if s in CLASS_TO_IDX else int(s)
    return int(x)

for df in (train_df, val_df):
    df["y"] = df[LABEL_COL].map(to_idx)

In [ ]:
from PIL import Image

for name, df in [("train", train_df), ("val", val_df)]:
    print(f"{name}: {len(df)} images | NORMAL={(df.y==0).sum()} | PNEUMONIA={(df.y==1).sum()} | patients={df[PID_COL].nunique()}")

common = set(train_df[PID_COL]) & set(val_df[PID_COL])
print("1. Patients communs train/val :", len(common)); assert len(common) == 0

allp = pd.concat([train_df, val_df])[PATH_COL]
missing = [p for p in allp if not os.path.exists(f"{DATA_ROOT}/{p}")]
print("2. Images manquantes :", len(missing)); assert len(missing) == 0, missing[:5]

for name, df in [("train", train_df), ("val", val_df)]:
    from_path = df[PATH_COL].str.contains("PNEUMONIA").astype(int)
    print(f"3. {name} incohérences label/dossier :", (from_path != df.y).sum())

print("4. Patients à 2 labels (train) :", (train_df.groupby(PID_COL).y.nunique() > 1).sum())
print("5. Doublons de chemin :", allp.duplicated().sum())

sample = pd.concat([train_df[train_df.y==0].sample(4, random_state=0),
                    train_df[train_df.y==1].sample(4, random_state=0)])
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, r) in zip(axes.ravel(), sample.iterrows()):
    ax.imshow(Image.open(f"{DATA_ROOT}/{r[PATH_COL]}").convert("L"), cmap="gray")
    ax.set_title(CLASSES[r.y]); ax.axis("off")
plt.show()

In [ ]:
print(train_df[train_df.y==1][[PATH_COL, PID_COL]].head(8))
print(train_df[train_df.y==1][PID_COL].nunique(), "patients PNEUMONIA distincts")

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torch.nn as nn

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class XrayDataset(Dataset):
    def __init__(self, df, root, tf):
        self.df, self.root, self.tf = df.reset_index(drop=True), root, tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(os.path.join(self.root, r[PATH_COL])).convert("RGB")
        return self.tf(img), int(r["y"])

In [ ]:
counts = train_df.y.value_counts().sort_index().values
class_weights = torch.tensor(len(train_df) / (2.0 * counts), dtype=torch.float32).to(device)
print("Poids [NORMAL, PNEUMONIA] :", class_weights)
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
from sklearn.metrics import roc_auc_score, confusion_matrix

def build_model(name):
    if name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, 2)
    elif name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        m.classifier = nn.Linear(m.classifier.in_features, 2)
    else:
        raise ValueError(name)
    return m.to(device)

@torch.no_grad()
def evaluate(model, loader, thr=0.5):
    model.eval()
    probs, labels, tot = [], [], 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        tot += criterion(out, y).item() * len(y)
        probs.append(torch.softmax(out, 1)[:, 1].cpu().numpy())
        labels.append(y.cpu().numpy())
    probs, labels = np.concatenate(probs), np.concatenate(labels)
    preds = (probs >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {"loss": tot/len(labels), "auc": roc_auc_score(labels, probs),
            "sensitivity": tp/(tp+fn), "specificity": tn/(tn+fp),
            "probs": probs, "labels": labels}

In [ ]:
def run_experiment(exp_id, arch, lr=1e-4, epochs=10, batch_size=32, patience=3):
    set_seed()
    model = build_model(arch)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    tl = DataLoader(XrayDataset(train_df, DATA_ROOT, train_tf), batch_size=batch_size,
                    shuffle=True, num_workers=2, pin_memory=True)
    vl = DataLoader(XrayDataset(val_df, DATA_ROOT, eval_tf), batch_size=batch_size,
                    shuffle=False, num_workers=2)
    best_auc, bad, best_res, history = -1, 0, None, []
    best_path = f"{CKPT_DIR}/{exp_id}_best.pt"
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train(); run_loss = 0.0
        for x, y in tl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = criterion(model(x), y)
            loss.backward(); opt.step()
            run_loss += loss.item() * len(y)
        r = evaluate(model, vl)
        h = {"epoch": ep, "train_loss": run_loss/len(train_df), "val_loss": r["loss"],
             "val_auc": r["auc"], "val_sens": r["sensitivity"], "val_spec": r["specificity"]}
        history.append(h)
        print(f"[{exp_id}] ep{ep} train_loss={h['train_loss']:.4f} val_loss={h['val_loss']:.4f} "
              f"AUC={h['val_auc']:.4f} sens={h['val_sens']:.3f} spec={h['val_spec']:.3f}")
        torch.save({"epoch": ep, "model": model.state_dict(), "opt": opt.state_dict()},
                   f"{CKPT_DIR}/{exp_id}_last.pt")
        pd.DataFrame(history).to_csv(f"{OUT_DIR}/{exp_id}_history.csv", index=False)
        if r["auc"] > best_auc:
            best_auc, bad, best_res = r["auc"], 0, r
            torch.save(model.state_dict(), best_path)
        else:
            bad += 1
            if bad >= patience:
                print("Arrêt anticipé."); break

    row = {"exp_id": exp_id, "arch": arch, "lr": lr, "batch": batch_size, "epochs_run": ep,
           "val_auc": round(best_auc, 4), "val_sens": round(best_res["sensitivity"], 4),
           "val_spec": round(best_res["specificity"], 4),
           "minutes": round((time.time()-t0)/60, 1)}
    path = f"{OUT_DIR}/results.csv"
    old = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    pd.concat([old, pd.DataFrame([row])]).to_csv(path, index=False)
    print("Essai enregistré :", row)
    return best_path

def plot_history(exp_id):
    h = pd.read_csv(f"{OUT_DIR}/{exp_id}_history.csv")
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="val")
    ax[0].set_title("Loss"); ax[0].legend()
    ax[1].plot(h.epoch, h.val_auc, label="AUC"); ax[1].plot(h.epoch, h.val_sens, label="sensibilité")
    ax[1].plot(h.epoch, h.val_spec, label="spécificité"); ax[1].set_title("Validation"); ax[1].legend()
    plt.savefig(f"{OUT_DIR}/{exp_id}_curves.png", dpi=150); plt.show()

In [ ]:
#run_experiment("test_rapide", "resnet50", epochs=2)
#plot_history("test_rapide")

In [ ]:
# run_experiment("E1_resnet50_lr1e-4", "resnet50", lr=1e-4)
# plot_history("E1_resnet50_lr1e-4")

In [ ]:
# run_experiment("E2_densenet121_lr1e-4", "densenet121", lr=1e-4)
# plot_history("E2_densenet121_lr1e-4")

In [ ]:
import sys
DELIV = os.path.dirname(glob.glob("/kaggle/input/**/best_model.pt", recursive=True)[0])
sys.path.insert(0, DELIV)
from model_utils import load_model, predict, get_target_layer

m, cfg = load_model(f"{DELIV}/best_model.pt", f"{DELIV}/config.json", device="cpu")
print("Architecture :", cfg["architecture"])
print("Couche Grad-CAM :", get_target_layer(m, cfg).__class__.__name__)

paths = val_df[PATH_COL].tolist()
probs = []
for i in range(0, len(paths), 32):
    imgs = [Image.open(f"{DATA_ROOT}/{p}") for p in paths[i:i+32]]
    probs.append(predict(imgs, m, cfg)[:, 1])
probs = np.concatenate(probs)

labels = val_df.y.values
print("AUC :", round(roc_auc_score(labels, probs), 4))
preds = (probs >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
print("Sensibilité :", round(tp/(tp+fn), 4), "| Spécificité :", round(tn/(tn+fp), 4))
print("Probabilités entre 0 et 1 :", bool(((probs >= 0) & (probs <= 1)).all()))